# 12 · Sampling-clock recovery

A digital receiver does not see a continuous waveform: its analog-to-digital
converter (ADC) takes samples at instants set by its own clock.  The symbols
were launched by the transmitter's digital-to-analog converter (DAC) on a
*different* clock.  Two oscillators never run at exactly the same rate: they
differ by a few to a hundred **parts per million** (ppm).  The receiver
therefore takes slightly more or fewer samples per symbol than it believes,
and the instants where it samples slide slowly through the symbols.

Frame synchronization ([11](11-frame-synchronization-and-timing.ipynb))
finds *one* timing offset for a whole capture; it cannot follow a timing
that keeps moving.  This notebook covers:

1. **where to sample**: the eye diagram and the best sampling instant;
2. the **sampling-clock offset**, and how it walks the samples out of the eye;
3. the **Gardner timing-error detector**: how three samples tell early from
   late, its S-curve, and its power form for small rolloffs;
4. the **clock-recovery loop**: tracking the drift, and the loop bandwidth
   that trades speed against jitter;
5. **resampling** onto the symbol grid, and why the equalizer cannot do this
   job for us;
6. removing a **known offset** measured on an earlier capture.

It builds on the eye diagrams and the matched filter of
[04](04-pulse-shaping-and-matched-filtering.ipynb) and the resampling of
[05](05-multirate-and-resampling.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RC, RRC, Constellation
from commkit.backend import to_device
from commkit.timing import Gardner

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
WIDE = np.multiply(plt.rcParams["figure.figsize"], (2, 1))


def host(a):
    return to_device(a, "cpu")


def link(*, ppm, rolloff=0.1, sps=2, num_symbols=2**17, delay=0.0, device=DEVICE):
    """16-QAM at 32 GBd, Es/N0 = 20 dB, a delay of `delay` samples and an ADC
    clock off by `ppm`, through the matched filter."""
    tx = ck.generate(
        C16,
        num_symbols=num_symbols,
        symbol_rate=32e9,
        sps=sps,
        pulse=RRC(rolloff=rolloff),
        rng=1,
        device=device,
    )
    rx = ck.impairments.apply_awgn(tx, esn0_db=20, rng=2)
    rx = ck.timing.fft_fractional_delay(rx, delay=delay)
    rx = ck.impairments.apply_sampling_clock_offset(rx, offset_ppm=ppm)
    return ck.filtering.matched_filter(rx)


def rc_wave(symbols, up=32, rolloff=0.5):
    """A real waveform of raised-cosine pulses, `up` points per symbol, unit
    amplitude at the symbol instants (an illustration, on the host)."""
    impulses = np.zeros(len(symbols) * up)
    impulses[::up] = symbols
    pulse = RC(rolloff=rolloff, span=8).taps(up)
    wave = np.convolve(impulses, pulse)[pulse.size // 2 :][: impulses.size]
    return np.arange(wave.size) / up, wave / pulse.max()


def block_snr(sig, block=4096):
    """SNR against the reference in consecutive blocks of symbols."""
    y, ref = sig.samples, sig.reference.symbols
    n = min(y.shape[-1], ref.shape[-1]) // block * block
    starts = np.arange(0, n, block)
    return starts + block / 2, [
        ck.metrics.snr(y[s : s + block], ref[s : s + block]) for s in starts
    ]

## 1. Where to sample: the eye

After the matched filter each symbol is a pulse that peaks at its own
instant and passes through zero at all the others' (the Nyquist criterion of
[04](04-pulse-shaping-and-matched-filtering.ipynb)).  Sampled exactly at the
symbol instants, the waveform takes the transmitted values; anywhere else it
is a mix of neighbouring symbols, **inter-symbol interference**.

The eye diagram overlays two symbol periods of the waveform, cut at the
symbol clock.  Below, the in-phase part of 16-QAM with a 0.1 rolloff, at 8
samples per symbol to show the waveform between samples.  The eye is open,
with four clean levels, only at the symbol instants (0, 1, 2); half a symbol
away the traces cross everywhere.  With a small rolloff the opening is
narrow: a timing error of a tenth of a symbol already smears the levels.

Coherent receivers typically sample at **2 samples per symbol**: one sample
at each symbol instant and one halfway between, at the crossing.  The
right-hand plot shows a stretch of the waveform with these samples:
filled dots on the symbols, open dots on the crossings.

In [ ]:
fine = link(ppm=0.0, sps=8, num_symbols=2**13)
fig, axes = plt.subplots(1, 2, figsize=WIDE)
ck.plotting.plot_eye_diagram(
    fine, quantity="i", ax=axes[0], title="Eye of 16-QAM (I), rolloff 0.1"
)
for t in (0, 1, 2):
    axes[0].axvline(t, color="w", linestyle=":", linewidth=1)

i_wave = host(fine.samples.real[8 * 100 : 8 * 116])
t_fine = np.arange(i_wave.size) / 8  # symbol periods
axes[1].plot(t_fine, i_wave, color="0.6", label="waveform")
axes[1].plot(t_fine[::8], i_wave[::8], "o", color="C0", label="symbol samples")
axes[1].plot(
    t_fine[4::8], i_wave[4::8], "o", mfc="none", color="C1", label="midpoint samples"
)
axes[1].set_xlabel("Time [symbol periods]")
axes[1].set_ylabel("In-phase amplitude")
axes[1].set_title("2 samples per symbol")
axes[1].legend(loc="upper right")
plt.show()

## 2. The sampling-clock offset

`apply_sampling_clock_offset` models an ADC whose clock runs at
`f_s · (1 + ε)` instead of `f_s`.  The samples still claim the nominal rate,
so the receiver takes `sps · (1 + ε)` samples per transmitted symbol, and its
sampling instants walk through the symbols by `ε` symbol periods per symbol.
The drift accumulates: after `1/(4ε)` symbols the samples sit a quarter of a
symbol off the eye's centre, after `1/(2ε)` on the crossings, and after
`1/ε` they have slipped by a whole symbol.

To make the walk visible, the first plot exaggerates the offset to 4 %
(40 000 ppm) and takes one sample per *receiver* symbol of a waveform of
alternating symbols `+1, -1, +1, ...`.  The first samples sit on the peaks,
the symbol instants (grey lines).  Each one lands 4 % of a symbol later than
the one before: after 6 symbols they sit a quarter of a symbol off, on the
slopes, after 12.5 symbols on the zero crossings, where they carry nothing,
and after 25 symbols on the peaks again, but one symbol behind.  The table shows how fast real offsets act at
32 GBd.

In [ ]:
eps = 0.04
t_alt, alternating = rc_wave((-1.0) ** np.arange(36))
k = np.arange(26)
t_rx = 4 + k * (1 + eps)  # where the receiver samples, in transmitter symbols
fig, ax = plt.subplots(figsize=WIDE * (1, 0.6))
ax.plot(t_alt, alternating, color="0.6")
for t in range(4, 32):
    ax.axvline(t, color="0.5", linewidth=0.6, zorder=0)
ax.plot(t_rx, np.interp(t_rx, t_alt, alternating), "o", color="C0")
ax.set_xlim(3.5, 31.5)
ax.set_xlabel("Time [transmitter symbol periods]")
ax.set_ylabel("In-phase amplitude")
ax.set_title("A clock 4 % fast: the samples walk off the symbol instants")
plt.show()

print("offset    quarter symbol      one symbol       (at 32 GBd)")
for ppm in (10, 50, 100):
    quarter, whole = 0.25 / (ppm * 1e-6), 1 / (ppm * 1e-6)
    print(
        f"{ppm:4d} ppm  {quarter:8.0f} symbols  {whole:8.0f} symbols  "
        f"= {whole / 32e9 * 1e6:5.2f} µs"
    )

At 50 ppm the eye moves by a quarter of a symbol in 5000 symbols, 0.16 µs at
32 GBd, while a typical capture holds 10⁵ to 10⁶ symbols.  Below, the same
16-QAM at 50 ppm: the eye diagram over three windows of 2000 symbols
starting at symbols 0, 2500 and 5000, cut at the receiver's nominal symbol
clock.  The open eye slides right, an eighth of a symbol per window, while
the receiver keeps sampling at 0, 1 and 2 (dotted line at 1).

In [ ]:
drift8 = link(ppm=50.0, sps=8, num_symbols=2**14)
fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2.4, 0.8))
)
for ax, start in zip(axes, (0, 2500, 5000), strict=True):
    window = drift8.samples[8 * start : 8 * (start + 2000)]
    ck.plotting.plot_eye_diagram(
        window, sps=8, quantity="i", ax=ax, title=f"symbols {start}-{start + 2000}"
    )
    ax.axvline(1, color="w", linestyle=":", linewidth=1)
plt.show()

The receiver keeps slicing at its nominal symbol instants.  The record used
for the rest of the notebook has 2¹⁷ symbols at 2 samples per symbol, 50 ppm,
the matched filter, then `decimate_to_symbol_rate`, which keeps every second
sample.  The constellations follow the eye: clean at first, blurred at an
eighth of a symbol of drift, a cloud at a quarter, and at half a symbol the
samples sit on the crossings, between the symbols.

In [ ]:
drifting = link(ppm=50.0)
sliced = ck.multirate.decimate_to_symbol_rate(drifting)
fig, axes = plt.subplots(
    1, 4, figsize=np.multiply(plt.rcParams["figure.figsize"], (2.4, 0.7))
)
for ax, (start, drift) in zip(
    axes, ((0, "0"), (2500, "1/8"), (5000, "1/4"), (10000, "1/2")), strict=True
):
    ck.plotting.plot_constellation(
        sliced.samples[start : start + 1000],
        ax=ax,
        title=f"drift {drift} symbol",
    )
plt.show()

## 3. The Gardner timing-error detector

To follow the drift, the receiver needs a measurement of its timing error
from the samples alone, without knowing the symbols.  At two samples per
symbol, Gardner's detector compares two successive symbol samples
`y_{k-1}`, `y_k` with the sample halfway between them, `m_k`:

$$e_k = \mathrm{Re}\{m_k\,(y_{k-1} - y_k)^*\}.$$

The idea is clearest at a transition between two opposite symbols.  The
waveform crosses zero halfway between them, so when the timing is right
`m_k = 0` and `e_k = 0`.  When the receiver samples **late**, the midpoint
has already moved towards `y_k`, so it has the sign of `y_k` and `e_k` is
negative; **early**, it still has the sign of `y_{k-1}` and `e_k` is positive.
The difference `y_{k-1} - y_k` makes the sign independent of the direction
of the transition.  Between two equal symbols there is no crossing,
`y_{k-1} - y_k ≈ 0`, and the detector says nothing.

Below, a binary waveform (raised-cosine pulses, rolloff 0.5) with a
falling and a rising transition, sampled a fifth of a symbol early, on time
and a fifth of a symbol late.

In [ ]:
t_wave, wave = rc_wave([1, 1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, 1, 1])

fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2.4, 0.75)), sharey=True
)
for ax, tau, name in zip(
    axes, (-0.2, 0.0, 0.2), ("early", "on time", "late"), strict=True
):
    ax.plot(t_wave, wave, color="0.6")
    for k_sym, color in ((5, "C0"), (9, "C1")):  # falling, rising transition
        t3 = np.array([k_sym - 1, k_sym - 0.5, k_sym]) + tau
        y_prev, m, y = np.interp(t3, t_wave, wave)
        e = m * (y_prev - y)
        ax.plot(t3[[0, 2]], [y_prev, y], "o", color=color)
        ax.plot(t3[1], m, "D", mfc="none", color=color)
        ax.annotate(f"e = {e:+.2f}", (t3[1], m), textcoords="offset points",
                    xytext=(8, 0), va="center", color=color)  # fmt: skip
    ax.axhline(0, color="0.85", linewidth=0.8, zorder=0)
    ax.set_xlim(2.5, 11.5)
    ax.set_title(f"{name} (τ = {tau:+.1f} symbol)")
    ax.set_xlabel("Time [symbol periods]")
axes[0].set_ylabel("Amplitude")
plt.show()

Dots are the symbol samples `y_{k-1}`, `y_k`, diamonds the midpoints `m_k`.
Both transitions give the same sign: positive early, zero on time,
negative late.  Here the crossings lie exactly halfway because the symbols
around each transition are symmetric; with other neighbours the crossing
moves a little, and `e` is not quite zero even on time.  The loop pushes the next sample later when `e > 0` and
earlier when `e < 0`.

### The S-curve

On random data, each `e_k` is noisy: half the time there is no transition,
and the neighbouring symbols disturb the midpoint (**pattern noise**).
Averaged over many symbols, `e` against the timing error `τ` is a sinusoid,
the **S-curve**.  Its zero at `τ = 0` is the lock point, and its slope there
is the detector gain `K_d`.  The other zero, half a symbol away, has the
opposite slope: there the loop is pushed away, so it never settles on the
crossings.

The classical detector relies on the transitions of the waveform, and a
pulse with a small rolloff makes them sluggish: its S-curve shrinks in
proportion to the excess bandwidth.  The **power** form applies the same
comparison to the instantaneous power `|x|²`,

$$e_k = -|m_k|^2\,(|y_{k-1}|^2 - |y_k|^2),$$

and draws on the different power levels of a multi-ring constellation such
as 16-QAM, which do not depend on the rolloff in the same way.  (It does not
work for QPSK, whose symbols all have the same power.)  Below, both S-curves
computed by hand: the matched-filter output delayed by known amounts,
normalized to unit power, split into symbol samples and midpoints.

In [ ]:
taus = np.linspace(-0.5, 0.5, 41)  # symbol periods
fig, axes = plt.subplots(1, 2, figsize=WIDE, sharey=True)
for rolloff, color in ((0.1, "C0"), (0.5, "C1")):
    mf = link(ppm=0.0, rolloff=rolloff, num_symbols=2**14)
    curves = {"classical": [], "power": []}
    for tau in taus:
        x = host(ck.timing.fft_fractional_delay(mf, delay=-2 * tau).samples)
        x = x / np.sqrt(np.mean(np.abs(x) ** 2))
        y, m = x[0::2], x[1::2]  # m[k] lies between y[k] and y[k + 1]
        curves["classical"].append(np.mean(np.real(m[:-1] * np.conj(y[:-1] - y[1:]))))
        p_y, p_m = np.abs(y) ** 2, np.abs(m) ** 2
        curves["power"].append(np.mean(-p_m[:-1] * (p_y[:-1] - p_y[1:])))
    for ax, (name, curve) in zip(axes, curves.items(), strict=True):
        ax.plot(taus, curve, color=color, label=f"rolloff {rolloff}")
        ax.set_title(f"{name} detector")
for ax in axes:
    ax.axhline(0, color="0.7", linewidth=0.8)
    ax.axvline(0, color="0.7", linewidth=0.8)
    ax.annotate("lock point", (0, 0), xytext=(0.12, 0.25), textcoords="data",
                arrowprops={"arrowstyle": "->", "color": "0.4"})  # fmt: skip
    ax.set_xlabel("Timing error τ [symbol periods]")
    ax.legend()
axes[0].set_ylabel("Mean detector output")
plt.show()

Both detectors are zero on time and negative when late.  At a rolloff of 0.5
the classical S-curve is the steeper; at 0.1 it has shrunk to a fifth, while
the power form is steeper at 0.1 than at 0.5.

`estimate_sampling_clock_offset` measures this S-curve itself before it
starts (open loop, over the first 2048 symbols) and reports its slope as
`detector_gain`.  Over the whole range of rolloffs, the classical gain grows
in proportion to the rolloff and is almost nothing without excess bandwidth;
the power detector is the stronger one up to a rolloff of about 0.3, the
classical one above.

In [ ]:
rolloffs = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0]
gains = {"classical": [], "power": []}
for rolloff in rolloffs:
    mf = link(ppm=0.0, rolloff=rolloff, num_symbols=2**13, delay=0.37)
    for method in gains:
        est = ck.timing.estimate_sampling_clock_offset(mf, Gardner(method=method))
        gains[method].append(float(est.detector_gain))
fig, ax = plt.subplots()
for method, g in gains.items():
    ax.plot(rolloffs, g, "o-", label=method)
ax.set_xlabel("RRC rolloff")
ax.set_ylabel("Detector gain $K_d$ [per symbol period]")
ax.set_title("Gardner detector gain, 16-QAM")
ax.legend()
plt.show()

## 4. The clock-recovery loop

`Gardner()` puts the detector in a feedback loop that sets *where* the next
sample is taken:

```
 samples ──► interpolator ──► y[k-1], m[k], y[k] ──► detector ──► e[k]
                  ▲                                                │
                  │       next strobe = this strobe                ▼
                  └──────    + sps · (1 + ν + μ·e[k])  ◄──── loop filter
                                                         ν ← ν + β·e[k]
```

* The **interpolator** computes the waveform between the ADC's samples (a
  windowed sinc), so the loop can sample at any instant, the **strobe**.
* The **loop filter** has two paths.  The proportional one, `μ·e`, nudges
  the next strobe; the integral one, `ν`, accumulates the error and learns
  the clock offset: once locked, `ν` settles at `ε` and the strobes advance
  by `sps · (1 + ε)` samples per symbol, just like the transmitter's symbols.
* The gains follow from one number, the **normalized loop bandwidth** `B·T`
  (`normalized_bandwidth`, default `1e-3`), and the measured `K_d`.  The
  open-loop S-curve also gives the right starting phase, so the loop does
  not need to search for it.

`estimate_sampling_clock_offset` runs the loop and returns the **timing
trajectory** `τ_k`: where the strobe of symbol `k` sits, in symbol periods,
minus `k`.  Without a clock offset it would be flat; here the ADC takes
`2(1 + ε)` samples per symbol, so the true trajectory is the ramp `k·ε`,
6.6 symbol periods over the record.  The offset in ppm is the slope of the
trajectory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=WIDE)
k = None
for method in ("classical", "power"):
    est = ck.timing.estimate_sampling_clock_offset(drifting, Gardner(method=method))
    traj = host(est.trajectory)
    k = np.arange(traj.size)
    print(
        f"{method:9s}: {float(est.value):7.3f} ppm (true 50), "
        f"detector gain {float(est.detector_gain):.2f}, "
        f"rms timing error {np.std(traj - k * 50e-6):.4f} symbol"
    )
    axes[0].plot(k, traj, label=method)
    axes[1].plot(k, traj - k * 50e-6, label=method, linewidth=0.8)
axes[0].plot(k, k * 50e-6, "--", color="0.5", label="true timing")
axes[0].set_ylabel("Timing τ [symbol periods]")
axes[0].set_title("Tracked timing")
axes[1].set_ylabel("Tracked minus true [symbol periods]")
axes[1].set_title("Timing error of the loop")
for ax in axes:
    ax.set_xlabel("Symbol")
    ax.legend()
plt.show()

Both loops follow the ramp; what remains is **jitter**, a hundredth of a
symbol or two: the detector's pattern noise (and, much less, the AWGN),
smoothed by the loop.  The power detector is a little noisier at this
rolloff even though its gain is larger.

### The loop bandwidth

`B·T` says how much of the detector's noise the loop lets through and how
fast it reacts.  Below, the power detector at three bandwidths, over the
first 40 000 symbols.  Each loop starts with `ν = 0`, unaware of the 50 ppm,
and must first learn it: the timing lags behind the ramp until the
integrator has caught up.

* `1e-4`: the jitter is smallest, but learning the offset takes some 30 000
  symbols, with a lag of almost a tenth of a symbol meanwhile;
* `1e-3` (default): locked within about a thousand symbols, with three times
  the jitter;
* `2e-2`: the loop reacts to every symbol, and its jitter becomes so large
  that it can be thrown past the edge of the S-curve into the next symbol, a
  **cycle slip**: the timing jumps by a whole symbol period (here three
  times, the first near symbol 7000), and every symbol after it is counted
  against the wrong reference.

A real clock also wanders (temperature, the ADC's own phase noise); the
bandwidth must be wide enough to follow that and narrow enough to keep the
jitter small.

The loop checks its own trajectory for slips and logs a WARNING when it
finds one, as below.  It also holds its integrator within
`max_offset_ppm` (default 1000 ppm, beyond any crystal clock): a loop that
has slipped feels no pull from the S-curve while its timing runs through the
symbols, and without that bound its frequency could wander off entirely.

A slip is a rare event that depends on the noise realization.  The library
draws channel noise on the data's device, so CPU and GPU records differ in
detail; this cell uses a record made on the CPU, so it shows the same slip
on every machine.

In [ ]:
on_cpu = link(ppm=50.0, device="cpu")
fig, axes = plt.subplots(1, 2, figsize=WIDE)
for bt, color in ((1e-4, "C0"), (1e-3, "C1"), (2e-2, "C2")):
    est = ck.timing.estimate_sampling_clock_offset(
        on_cpu, Gardner(method="power", normalized_bandwidth=bt)
    )
    traj = host(est.trajectory)
    err = traj - np.arange(traj.size) * 50e-6
    print(
        f"B·T = {bt:.0e}: {float(est.value):8.3f} ppm, rms error after symbol "
        f"20000: {np.std(err[20000:]):.4f} symbol, largest error {np.max(np.abs(err)):.2f}"
    )
    ax = axes[1] if bt == 2e-2 else axes[0]
    ax.plot(err[:40000], color=color, label=f"B·T = {bt:.0e}", linewidth=0.8)
axes[0].set_ylim(-0.15, 0.1)
axes[0].set_title("Acquisition and jitter")
axes[1].set_title("Too wide: cycle slips")
for ax, loc in zip(axes, ("lower right", "center right"), strict=True):
    ax.set_xlabel("Symbol")
    ax.legend(loc=loc)
axes[0].set_ylabel("Tracked minus true [symbol periods]")
plt.show()

## 5. Resampling onto the symbol grid

`correct_sampling_clock_offset` with a `Gardner` runs the same loop and then
resamples the record: `sps` samples per tracked symbol, starting at each
strobe, with the band-limited interpolator of `apply_sampling_clock_offset`.
The output is at 2 samples per symbol again, but on the *transmitter's*
clock: the drift and the static timing are both gone, and symbol `k` of the
output is the `k`-th transmitted symbol, so the reference stays aligned.

Below, the symbol-rate SNR in blocks of 4096 symbols along the record.
Without clock recovery it collapses after a few thousand symbols and never
recovers: the samples first leave the eye, then slip onto the neighbouring
symbols.  After recovery it is flat over the whole record.

The constellation of the whole record without recovery still shows the 16
points, blurred: once per symbol of drift the samples pass through the eye's
centre again, but on the wrong symbols, which is what the SNR against the
reference sees.

In [ ]:
fig, ax = plt.subplots(figsize=WIDE * (1, 0.7))
centre, snr = block_snr(sliced)
ax.plot(centre, snr, "o-", color="0.5", label="no clock recovery")
for method in ("classical", "power"):
    recovered = ck.timing.correct_sampling_clock_offset(
        drifting, Gardner(method=method)
    )
    sym = ck.multirate.decimate_to_symbol_rate(recovered)
    centre, snr = block_snr(sym)
    ax.plot(centre, snr, "o-", label=f"Gardner, {method}")
    print(f"{method:9s}: SNR {ck.metrics.snr(sym, num_skip_symbols=1000):.2f} dB")
ax.axhline(20, color="0.7", linestyle=":", label="Es/N0 of the channel")
ax.set_xlabel("Symbol")
ax.set_ylabel("SNR per block [dB]")
ax.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=WIDE)
ck.plotting.plot_constellation(sliced, ax=axes[0], title="Without clock recovery")
ck.plotting.plot_constellation(sym, ax=axes[1], title="After clock recovery")
plt.show()

The recovered SNR is a fraction of a dB below the channel's 20 dB.  The
remaining jitter leaves ISI of power `σ_τ² · Σ_{n≠0} g'(n)²`, with `σ_τ` the
rms timing error and `g'` the slope of the raised-cosine pulse at the other
symbols' instants.  A smaller rolloff has steeper tails, so the same jitter
costs it more.

### Why not leave it to the equalizer?

An adaptive equalizer ([17](17-adaptive-equalization.ipynb)) also corrects
timing: its taps interpolate between samples.  But it has a fixed window of
`num_taps / sps` symbols, and a drift moves the symbol out of it.  Below,
trained LMS with 11 taps (a window of 5.5 symbols) on the drifting record
and on the recovered one.  Its output SNR, in blocks: without clock
recovery the equalizer follows the drift until the symbol leaves its
window, after 2-3 symbols of drift (about 50 000 symbols here), and fails
from then on; `lms` warns that its decisions no longer have the
constellation's power.

In [ ]:
train = drifting.reference.symbols[..., :4000]
kw = {"num_taps": 11, "step_size": 1e-3}
fig, ax = plt.subplots(figsize=WIDE * (1, 0.7))
for name, rx in (
    ("LMS alone", drifting),
    ("Gardner, then LMS", ck.timing.correct_sampling_clock_offset(drifting, Gardner())),
):
    eq = ck.equalization.lms(rx, train, **kw)
    centre, snr = block_snr(eq.signal)
    ax.plot(centre, snr, "o-", label=name)
ax.set_xlabel("Symbol")
ax.set_ylabel("SNR per block [dB]")
ax.set_title("Trained LMS, 11 taps, 50 ppm clock offset")
ax.legend()
plt.show()

## 6. A known offset

The clock offset of a setup changes slowly, so an estimate from one capture
serves the next.  `correct_sampling_clock_offset` with a
`SamplingClockOffsetEstimate` resamples by the constant factor `1 + ε`: the
drift goes, a static timing phase stays, and the adaptive equalizer absorbs
that phase with the rest of the channel.  Below, the estimate of the record
above applied to a new capture with a different delay; a second estimate on
the corrected capture finds almost no offset left.

In [ ]:
est = ck.timing.estimate_sampling_clock_offset(drifting, Gardner(method="power"))
next_capture = link(ppm=50.0, delay=1.1)
fixed = ck.timing.correct_sampling_clock_offset(next_capture, est)
residual = ck.timing.estimate_sampling_clock_offset(fixed, Gardner(method="power"))
print(f"offset of the first capture: {float(est.value):.3f} ppm")
print(f"residual offset after the static correction: {float(residual.value):.3f} ppm")

Two polarizations sampled by the same ADC clock share one offset: with
`Gardner(joint_channels=True)` a single loop averages the detector outputs
of both channels and gives them one trajectory.

**Summary**

* An ADC clock off by `ε` walks the sampling instants through the symbols by
  `ε` symbol periods per symbol; at tens of ppm a capture of 10⁵ symbols
  drifts by whole symbols.
* Gardner's detector compares two symbol samples with the midpoint between
  them; it needs two samples per symbol and transitions.  The classical form
  needs excess bandwidth, the power form works at small rolloffs for
  multi-ring constellations.
* The loop's bandwidth trades acquisition speed and tracking against jitter;
  too wide, it slips whole symbols.
* `estimate_sampling_clock_offset` measures the offset and the timing
  trajectory; `correct_sampling_clock_offset` resamples onto the symbol grid,
  tracking with `Gardner()` or removing a known offset.  An equalizer cannot
  replace it: a drift leaves its window.

**Next:** [13 · Frequency-offset estimation](13-frequency-offset-estimation.ipynb).